In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Câu 1:

In [23]:
df_orders= pd.read_csv('../data/orders.csv', parse_dates=['order_date'])

In [24]:
df_orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [25]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 39.5+ MB


In [26]:
customer_count= df_orders.groupby('customer_id').size()
multi_order_customers = customer_count[customer_count >= 2]
print(f"Số lượng khách hàng đã đặt hàng nhiều lần: {len(multi_order_customers)}")

Số lượng khách hàng đã đặt hàng nhiều lần: 67888


In [27]:
orders_multi= df_orders[df_orders['customer_id'].isin(multi_order_customers.index)]
orders_multi= orders_multi.sort_values(by=['customer_id', 'order_date'])
orders_multi['prev_date']= orders_multi.groupby('customer_id')['order_date'].shift(1)
orders_multi['days_between']= (orders_multi['order_date'] - orders_multi['prev_date']).dt.days
median_between= orders_multi['days_between'].median()
print(f"Thời gian trung bình giữa các đơn hàng của khách hàng: {median_between:.2f} ngày") 

Thời gian trung bình giữa các đơn hàng của khách hàng: 144.00 ngày


### ==> Đáp án: C

## Câu 2:

In [28]:
df_products= pd.read_csv('../data/products.csv')

In [29]:
df_products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [31]:
df_products['segment'].unique()

array(['Everyday', 'Performance', 'Balanced', 'Standard', 'All-weather',
       'Premium', 'Trendy', 'Activewear'], dtype=object)

In [32]:
gross_margin= (df_products['price']- df_products['cogs'])/ df_products['price']
df_products['gross_margin'] = gross_margin
segment_margin= df_products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print("Lợi nhuận gộp trung bình theo phân khúc sản phẩm:")
print(segment_margin)

Lợi nhuận gộp trung bình theo phân khúc sản phẩm:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


### ==> Đáp án: D

## Câu 3:

In [34]:
df_returns= pd.read_csv('../data/returns.csv')
df_merge= pd.merge(df_products, df_returns, on='product_id', how='left')

In [36]:
df_merge.head()

,product_id,product_name,category,segment,size,color,price,cogs,gross_margin,return_id,order_id,return_date,return_reason,return_quantity,refund_amount
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225,RET-017121,268094.0,2015-04-21,not_as_described,2.0,17631.46
1,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225,RET-017720,277786.0,2015-05-19,wrong_size,8.0,87302.64
2,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336,RET-034568,549673.0,2017-12-20,changed_mind,1.0,6580.77
3,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336,RET-035665,566897.0,2018-03-31,wrong_size,1.0,8969.16
4,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336,RET-037079,590682.0,2018-06-12,defective,5.0,46022.01


In [45]:
df_streetwear= df_merge[df_merge['category'] == 'Streetwear']
most_return_reason= df_streetwear.groupby('return_reason').size().sort_values(ascending=False)
print("Lý do trả hàng phổ biến nhất của Streetwear:", most_return_reason.index[0])

Lý do trả hàng phổ biến nhất của Streetwear: wrong_size


### ==> Đáp án: B

## Câu 4:

In [46]:
df_web_traffic= pd.read_csv('../data/web_traffic.csv', parse_dates=['date'])

In [47]:
df_web_traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [49]:
print(df_web_traffic['traffic_source'].unique())

['organic_search' 'direct' 'referral' 'social_media' 'paid_search'
 'email_campaign']


In [51]:
avg_bound_rate= df_web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values(ascending=True)
print("Tỷ lệ thoát trung bình theo nguồn truy cập:")
print(avg_bound_rate)

Tỷ lệ thoát trung bình theo nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


### ==> Đáp án: C

## Câu 5:

In [54]:
df_order_items= pd.read_csv('../data/order_items.csv', low_memory=False)

In [55]:
df_order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [56]:
total_row= len(df_order_items)
print(f"Tổng số dòng trong order_items: {total_row}")

Tổng số dòng trong order_items: 714669


In [60]:
is_promo= df_order_items['promo_id'].notna() | df_order_items['promo_id_2'].notna()
print(f"Số lượng dòng có promo_id hoặc promo_id_2 không rỗng:", len(df_order_items[is_promo]))

Số lượng dòng có promo_id hoặc promo_id_2 không rỗng: 276316


In [61]:
print("Tỷ lệ phần trăm áp dụng khuyến mãi:", len(df_order_items[is_promo]) / total_row * 100)

Tỷ lệ phần trăm áp dụng khuyến mãi: 38.663493169565214


### ==> Đáp án: C

## Câu 6:

In [62]:
df_customers= pd.read_csv('../data/customers.csv')

In [64]:
df_customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search
